### TRANSFORMATION WORKFLOW

1. JOIN CRM + ERP CUSTOMERS + ERP LOCATIONS
2. ADD SURROGATE KEY
3. WRITE INTO GOLD DIMENSION TABLE

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StringType 
from pyspark.sql.window import Window

In [0]:
# 0) LOAD ALL THE CUSTUMER RELATED TABLES FROM SILVER LAYER
df_crmc = spark.table("acdproj.silver.crm_customers")
df_erpc = spark.table("acdproj.silver.erp_customers")
df_erploc = spark.table("acdproj.silver.erp_locations")

In [0]:
# 1) JOIN CRM + ERP CUSTOMERS + ERP LOCATIONS
dim_customers = (
    df_crmc.alias("crm")
    .join(df_erpc.alias("erp"), F.col("crm.customer_key") == F.col("erp.customer_id"), "left")
    .join(df_erploc.alias("loc"), F.col("crm.customer_key") == F.col("loc.customer_id"), "left")
    .select(
        F.col("crm.customer_key").alias("customer_key"),
        F.col("crm.firstname"),
        F.col("crm.lastname"),
        F.col("crm.marital_status"),
        F.when(F.col("crm.customer_gender") != "n/a", F.col("crm.customer_gender"))
         .otherwise(F.col("erp.customer_gender"))
         .alias("customer_gender"),
        F.col("crm.create_date"),
        F.col("erp.birth_date"),
        F.col("loc.country")
    )
)


In [0]:
# 2) ADD SURROGATE KEY
window_spec = Window.orderBy("customer_key")
dim_customers = dim_customers.withColumn("customer_sk", F.row_number().over(window_spec))

# Reorder so the surrogate key appears first (optional, just for readability)
dim_customers = dim_customers.select(
    "customer_sk", "customer_key", "firstname", "lastname",
    "marital_status", "customer_gender", "birth_date", "country", "create_date"
)


In [0]:
# 3) WRITE INTO GOLD
dim_customers.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("acdproj.gold.dim_customers")